# 🐍 Clase 9 · Primera estrategia + métricas

> Cerrar el motor: una estrategia con señal real, y métricas honestas — PnL, posición y slippage contra el mid de llegada. Checkpoint integrador de todo L1-L9.

**Hoy construyes:** medir una estrategia contra un benchmark.

### Cómo funciona este cuaderno

1. Escribe tu respuesta en la celda de código.
2. Debajo hay una **✅ comprobación plegada**: ejecútala con `Shift+Enter` para validarte (despliégala si quieres ver el `assert`).
3. ¿Atascado? Abre **💡 Ver solución**.

**Núcleo:** los primeros (en clase) · **Si vamos bien:** el resto · **Más:** el cuaderno de auxiliares.

### 1. Estrategia de imbalance

Define `ImbalanceStrategy(thr)`: compra 0.05 si `imbalance(3) > thr`, vende 0.05 si `< -thr`. Market orders.

<sub>practicas: señal larga/corta</sub>

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class ImbalanceStrategy(Strategy):
    def __init__(self, thr=0.3):
        self.thr = thr
    def on_book_update(self, book):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
r = Backtest(Market.sample(), ImbalanceStrategy(0.3)).run()
assert r.n_steps == 500
print('ok ', r)

<details>
<summary>💡 Ver solución</summary>

```python
class ImbalanceStrategy(Strategy):
    def __init__(self, thr=0.3):
        self.thr = thr
    def on_book_update(self, book):
        imb = book.imbalance(3)
        if imb is None: return []
        if imb > self.thr:
            return [NewOrder(Order('BTCUSDT', Side.BUY, 0.05, order_type=OrderType.MARKET))]
        if imb < -self.thr:
            return [NewOrder(Order('BTCUSDT', Side.SELL, 0.05, order_type=OrderType.MARKET))]
        return []
```

</details>

### 2. Mídela

Corre la estrategia y guarda `equity`, `pos` y `fills` del resultado.

<sub>practicas: leer BacktestResult</sub>

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class ImbalanceStrategy(Strategy):
    def __init__(self, thr=0.3): self.thr=thr
    def on_book_update(self, book):
        imb = book.imbalance(3)
        if imb is None: return []
        if imb > self.thr: return [NewOrder(Order('BTCUSDT', Side.BUY, 0.05, order_type=OrderType.MARKET))]
        if imb < -self.thr: return [NewOrder(Order('BTCUSDT', Side.SELL, 0.05, order_type=OrderType.MARKET))]
        return []
equity = None
pos = None
fills = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert isinstance(equity, float) and isinstance(fills, int)
print('ok  equity=%.2f pos=%.3f fills=%d' % (equity, pos, fills))

<details>
<summary>💡 Ver solución</summary>

```python
r = Backtest(Market.sample(), ImbalanceStrategy(0.3)).run()
equity = r.final_equity
pos = r.final_position
fills = r.n_fills
```

</details>

### 3. Benchmark de llegada

Guarda `arrival_mid`, el mid del primer snapshot del mercado.

<sub>practicas: mid inicial</sub>

In [ ]:
from exchange import Market
arrival_mid = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert 90000 < arrival_mid < 110000
print('ok  arrival_mid=%.2f' % arrival_mid)

<details>
<summary>💡 Ver solución</summary>

```python
from exchange import Market
arrival_mid = Market.sample().step().mid
```

</details>

### 4. Riesgo escondido

Para `thr=0.1` y `thr=0.5`, guarda en `pos_small_thr` y `pos_big_thr` la posición final. Un umbral bajo opera más y arriesga más inventario.

<sub>practicas: interpretar inventario</sub>

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class ImbalanceStrategy(Strategy):
    def __init__(self, thr=0.3): self.thr=thr
    def on_book_update(self, book):
        imb = book.imbalance(3)
        if imb is None: return []
        if imb > self.thr: return [NewOrder(Order('BTCUSDT', Side.BUY, 0.05, order_type=OrderType.MARKET))]
        if imb < -self.thr: return [NewOrder(Order('BTCUSDT', Side.SELL, 0.05, order_type=OrderType.MARKET))]
        return []
pos_small_thr = None
pos_big_thr = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert pos_small_thr is not None and pos_big_thr is not None
print('ok  thr0.1 pos=%.3f | thr0.5 pos=%.3f' % (pos_small_thr, pos_big_thr))

<details>
<summary>💡 Ver solución</summary>

```python
pos_small_thr = Backtest(Market.sample(), ImbalanceStrategy(0.1)).run().final_position
pos_big_thr = Backtest(Market.sample(), ImbalanceStrategy(0.5)).run().final_position
```

</details>

## Cierre

Sin benchmark no hay estrategia: medir contra el mid de llegada separa la suerte del valor.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.